<a href="https://colab.research.google.com/github/illusoryTwin/mujoco_unitree_a1/blob/menagerie_model/code_tests/UnitreeA1_v2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation

In [1]:
!pip install mujoco

# Set up GPU rendering.
from google.colab import files
import distutils.util
import os
import subprocess
# if subprocess.run('nvidia-smi').returncode:
#   raise RuntimeError(
#       'Cannot communicate with GPU. '
#       'Make sure you are using a GPU Colab runtime. '
#       'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Other imports and helper functions
import numpy as np
from typing import Callable, Optional, Union, List
import scipy.linalg

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()


In [2]:
print("Get the unitree a1 model")
!git clone https://github.com/deepmind/mujoco_menagerie

Get the unitree a1 model
Cloning into 'mujoco_menagerie'...
remote: Enumerating objects: 2474, done.
remote: Counting objects: 100% (816/816), done.
remote: Compressing objects: 100% (524/524), done.
remote: Total 2474 (delta 405), reused 540 (delta 291), pack-reused 1658 (from 1)
Receiving objects: 100% (2474/2474), 227.13 MiB | 13.18 MiB/s, done.
Resolving deltas: 100% (955/955), done.
Updating files: 100% (1336/1336), done.


# Auxiliary functions

### Note: we will use XYZ Euler angles representation

Formulas with quaternions:
https://stengel.mycpanel.princeton.edu/Quaternions.pdf

In [10]:
from scipy.spatial.transform import Rotation as R


class Calculator:
  def __init__(self, model, data):
  # def __init__(self, data, model, qpos_=data.qpos):
    self.model = model
    self.data = data

  def calc_euler_angles(self, qpos_):
    # TO DO: rewrite the comment
    '''Function for recalculation of data.qpos based on quaternions into
    new_data_qpos with euler angles'''

    quat = qpos_[3:7]
    r = R.from_quat([quat[1], quat[2], quat[3], quat[0]])
    euler_angles = r.as_euler('xyz', degrees=False)
    return euler_angles

  def get_qpos_with_euler_angles(self, qpos_):
    # TO DO: rewrite the comment
    '''Function for recalculation of data.qpos based on quaternions into
    new_data_qpos with euler angles'''

    quat = qpos_[3:7]
    r = R.from_quat([quat[1], quat[2], quat[3], quat[0]])
    euler_angles = r.as_euler('xyz', degrees=False)
    euler_angles_data_qpos = np.concatenate((qpos_[0:3], euler_angles, qpos_[7:]))
    return euler_angles_data_qpos


  def get_qpos_with_quaternions(self, qpos_):
    euler_angles = qpos_[3:6]
    r = R.from_euler('xyz', euler_angles, degrees=False) # v3.0
    quat = r.as_quat()
    quat_mujoco = [quat[1], quat[2], quat[3], quat[0]]
    quat_data_qpos = np.concatenate((qpos_[0:3], quat_mujoco, qpos_[6:]))
    return quat_data_qpos

  def calc_angle_derivs(self, qpos_):
    '''Function that calculates time derivatives of Euler angles. For the
    formula, check the reference above'''
    euler_angles_ = self.calc_euler_angles(qpos_)
    omega_x = self.data.qvel[3]
    omega_y = self.data.qvel[4]
    omega_z = self.data.qvel[5]

    omega = np.array([omega_x, omega_y, omega_z])

    phi = euler_angles_[0]
    theta = euler_angles_[1]
    psi = euler_angles_[2]

    rot_matrix = np.array([
        [1, np.sin(phi)*np.tan(theta), np.cos(phi)*np.tan(theta)],
        [0, np.cos(phi), -np.sin(phi)],
        [0, np.sin(phi)/np.cos(theta), np.cos(phi)/np.cos(theta)]
    ])

    angle_derivs = np.dot(rot_matrix, omega)

    return angle_derivs

  def get_velocities_vector(self, qpos_):
    '''Function that creates a qelocity vector based on data.qvel with the euler
    angles derivatives'''
    angle_derivs = self.calc_angle_derivs(qpos_)
    return np.concatenate(self.data.qvel[0:3].copy(), angle_derivs, self.data.qvel[6:].copy())

  def get_qvel_standard(self, euler_angles_derivs):
    rot_matrix_inv = np.inverse(np.array([
        [1, np.sin(phi)*np.tan(theta), np.cos(phi)*np.tan(theta)],
        [0, np.cos(phi), -np.sin(phi)],
        [0, np.sin(phi)/np.cos(theta), np.cos(phi)/np.cos(theta)]
    ]))
    angle_vels = np.dot(rot_matrix_inv, euler_angles_derivs)
    return angle_vels

  def get_state_vector(self, qpos_):
    '''Function that composes a state vector X, which consists of the state
    in joint coordinates and its time derivative'''
    # qpos_euler = self.get_qpos_with_euler_angles(qpos_)
    qvel_euler = self.get_velocities_vector(qpos_)
    return qpos_euler
    # return np.concatenate((qpos_euler, qvel_euler))

  def get_d_state_vector():
    '''Function that composes a state vector X, which consists of the 1st and
    2nd order time derivatives of state in joint coordinates'''
    pass


In [13]:
model = mujoco.MjModel.from_xml_path("mujoco_menagerie/unitree_a1/scene.xml")
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model)

duration = 0.1  # (seconds)
framerate = 60  # (Hz)

frames = []
data.qpos = [0] * 19

while data.time < duration:
  qpos_init = [ 0., # x pos
                0., # y pos
                0.43, # z pos

                1.0,  # w quaternion component
                0.3, # x component
                0., # y component
                0.9, # z component

                0., # FR_hip, front right
                0., # FR_thigh
                0., # FR_calf
                0., # FL_hip, front left
                0., # FL_thigh
                0., # FL_calf
                0., # RR_hip, rear right
                0., # RR_thigh
                0., # RR_calf
                0., # RL_hip, rear left
                0., # RL_thigh
                0.  # RL_calf
                ]

  calc = Calculator(model, data)
  data.qpos = qpos_init
  res = calc.calc_angle_derivs(qpos_init)
  # X = calc.get_state_vector(qpos_init)

  print("X", X)

  # euler_data_qpos = calc.get_qpos_with_euler_angles(qpos_init)
  # data_qpos_quat_back = calc.get_qpos_with_quaternions(euler_data_qpos)

  # data.qpos = data_qpos_quat_back

  # angle_derivs = calc.calc_angle_derivs(qpos_init)

  # if data.time == 0:
  #   print("euler_data_qpos", euler_data_qpos)
  #   print("data_qpos_quat_back", data_qpos_quat_back)
  #   print("data.qpos          ", data.qpos)

  # data.qpos = qpos_init
  mujoco.mj_step(model, data)

  if len(frames) < data.time * framerate:
      renderer.update_scene(data)
      pixels = renderer.render()
      frames.append(pixels)

media.show_video(frames, fps=framerate)


X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.   ]
X [ 0.     0.     0.43   0.336 -0.288  1.416  0.     0.     0.     0.     0.     0.     0.     0.
  0.  

In [ ]:
#Test

from scipy.spatial.transform import Rotation as R
import numpy as np
import mujoco
import mujoco_viewer

class Calculator:
    def __init__(self, model, data):
        self.model = model
        self.data = data

    def calc_euler_angles(self, qpos_):
        '''Function to calculate Euler angles from quaternions in data.qpos'''
        quat = qpos_[3:7]
        r = R.from_quat([quat[1], quat[2], quat[3], quat[0]])
        euler_angles = r.as_euler('xyz', degrees=False)
        return euler_angles

    def get_qpos_with_euler_angles(self, qpos_):
        '''Function to recalculate data.qpos based on quaternions into new data.qpos with Euler angles'''
        euler_angles = self.calc_euler_angles(qpos_)
        euler_angles_data_qpos = np.concatenate((qpos_[0:3], euler_angles, qpos_[7:]))
        return euler_angles_data_qpos

    def get_qpos_with_quaternions(self, qpos_):
        '''Function to recalculate data.qpos with Euler angles back into quaternions'''
        euler_angles = qpos_[3:6]
        r = R.from_euler('xyz', euler_angles, degrees=False)
        quat = r.as_quat()
        quat_mujoco = [quat[1], quat[2], quat[3], quat[0]]
        quat_data_qpos = np.concatenate((qpos_[0:3], quat_mujoco, qpos_[6:]))
        return quat_data_qpos

    def calc_angle_derivs(self, qpos_):
        '''Function to calculate the derivatives of the Euler angles'''
        euler_angles_ = self.calc_euler_angles(qpos_)
        omega_x = self.data.qvel[3]
        omega_y = self.data.qvel[4]
        omega_z = self.data.qvel[5]

        omega = np.array([omega_x, omega_y, omega_z])

        phi = euler_angles_[0]
        theta = euler_angles_[1]
        psi = euler_angles_[2]

        rot_matrix = np.array([
            [1, np.sin(phi)*np.tan(theta), np.cos(phi)*np.tan(theta)],
            [0, np.cos(phi), -np.sin(phi)],
            [0, np.sin(phi)/np.cos(theta), np.cos(phi)/np.cos(theta)]
        ])

        angle_derivs = np.dot(rot_matrix, omega)

        return angle_derivs

    def get_velocities_vector(self, qpos_):
        '''Function to get velocity vectors with Euler angle derivatives'''
        angle_derivs = self.calc_angle_derivs(qpos_)
        return np.concatenate((self.data.qvel[0:3].copy(), angle_derivs, self.data.qvel[6:].copy()))

    def get_state_vector(self, qpos_):
        '''Function to get the state vector with Euler angles'''
        qpos_euler = self.get_qpos_with_euler_angles(qpos_)
        qvel_euler = self.get_velocities_vector(qpos_)
        return np.concatenate((qpos_euler, qvel_euler))

    def get_d_state_vector(self, qpos_):
        pass

# Assuming your mujoco model is correctly set up and you have the necessary mujoco and viewer modules installed
model = mujoco.MjModel.from_xml_path("mujoco_menagerie/unitree_a1/scene.xml")
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model)

duration = 0.1  # (seconds)
framerate = 60  # (Hz)

frames = []
data.qpos = [0] * 19

while data.time < duration:
    qpos_init = [
        0.,  # x pos
        0.,  # y pos
        0.43,  # z pos
        1.0,  # w quaternion component
        0.3,  # x component
        0.,  # y component
        0.9,  # z component
        0.,  # FR_hip, front right
        0.,  # FR_thigh
        0.,  # FR_calf
        0.,  # FL_hip, front left
        0.,  # FL_thigh
        0.,  # FL_calf
        0.,  # RR_hip, rear right
        0.,  # RR_thigh
        0.,  # RR_calf
        0.,  # RL_hip, rear left
        0.,  # RL_thigh
        0.  # RL_calf
    ]

    calc = Calculator(model, data)
    data.qpos = qpos_init
    X = calc.get_state_vector(qpos_init)

    print("X", X)

    mujoco.mj_step(model, data)

    if len(frames) < data.time * framerate:
        renderer.update_scene(data)
        pixels = renderer.render()
        frames.append(pixels)

media.show_video(frames, fps=framerate)


ModuleNotFoundError: No module named 'mujoco_viewer'